# MLP Hyperparameter Tuning — Optuna Bayesian Optimisation

## What Optuna does differently

The sequential (OFAT) notebook sweeps one parameter at a time, greedily fixing each best value
before moving on. This misses **interaction effects** — e.g. a wider architecture may need
a different regularisation strength than a narrow one.

Optuna uses **Tree-structured Parzen Estimators (TPE)**, a form of Bayesian optimisation:

| Step | What happens |
|---|---|
| Trial 1–10 | Random warm-up — explores the space broadly |
| Trial 11+ | Fits a probabilistic model of which regions are promising |
| Each new trial | Samples from the *high-probability-good* region, not randomly |

This means 100 Optuna trials typically outperform 300 random trials, because the sampler
is intelligently concentrating effort where the loss surface is low.

**Objective**: minimise CV NMAD ($1.4826 \times \text{median}|\hat{y} - y|$) over the full
joint hyperparameter space in one shot.

In [ ]:
# Install optuna if not already present
import importlib, subprocess, sys
if importlib.util.find_spec('optuna') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])
print('optuna ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from astropy.stats import sigma_clip
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
)
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial spam

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

---
## Data Loading & Quality Cuts

Identical pipeline to `multilayer perceptron learning.ipynb` — same cuts, same train/test split,
same random seed so results are directly comparable.

In [ ]:
df = pd.read_csv('ZTF_DESI_ml_work/ZTF_DESI_data/ZTF_resid_cent_hostprop_no_x1_c.csv')
print(len(df), 'SNe before quality cuts.')

df = df[(df['lccoverage_flag'] == 1) & (df['fitquality_flag'] == 1)]
print(len(df), 'SNe after lccoverage + fitquality flags.')

df['SDSS_g_minus_r'] = df['ABSMAG01_SDSS_G'] - df['ABSMAG01_SDSS_R']

clip = sigma_clip(df['residual_centered'], sigma=3, maxiters=1)
df = df.loc[~clip.mask].reset_index(drop=True)
print(len(df), 'SNe after 3-sigma clip.')

df = df.loc[df['SFR'] <= 2.5].reset_index(drop=True)
df = df.loc[df['DN4000'] >= 0.5].reset_index(drop=True)
df = df.loc[df['AGE'] >= 2].reset_index(drop=True)
print(len(df), 'SNe after physical host-galaxy cuts.')

yerr_all = df['sigma_mu_meas'].copy()

In [ ]:
feature_cols = [
    'LOGMSTAR', 'c', 'x1', 'SFR', 'VDISP',
    'DN4000', 'SDSS_g_minus_r', 'AGE', 'redshift',
]
target_col = 'residual_centered'

X = df[feature_cols].copy()
y = df[target_col].copy()

Xtr, Xte, ytr, yte, yerr_tr, yerr_te = train_test_split(
    X, y, yerr_all, test_size=0.2, random_state=SEED
)
print(f'Training : {len(Xtr)} samples')
print(f'Test     : {len(Xte)} samples')

---
## Helper Functions

In [ ]:
def nmad(y_true, y_pred):
    """Normalised median absolute deviation — standard SN cosmology scatter metric."""
    return 1.4826 * float(np.median(np.abs(np.asarray(y_pred) - np.asarray(y_true))))


def outlier_rate(y_true, y_pred, threshold=0.1):
    """Fraction of predictions with |residual| > threshold mag."""
    return 100.0 * float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) > threshold))


def cv_scores(pipe, X, y, cv=5):
    """Return (NMAD, outlier_rate) via cross_val_predict."""
    yp = cross_val_predict(pipe, X, y, cv=cv)
    return nmad(y, yp), outlier_rate(y, yp)


print('Helpers defined.')

---
## Optuna Objective Function

The objective is the function Optuna minimises. Each call:
1. Receives a `trial` object from Optuna's TPE sampler
2. Uses `trial.suggest_*` to sample one value per hyperparameter
3. Builds and cross-validates the pipeline
4. Returns the CV NMAD — Optuna records this and updates its internal model

### Conditional hyperparameters

`learning_rate_init` is only sampled when `solver='adam'` — `lbfgs` does its own
internal line search and ignores this parameter. Optuna handles conditional sampling
correctly: it learns that when `solver=lbfgs`, `learning_rate_init` is irrelevant
and does not waste sampling budget on it.

In [ ]:
# Architecture candidates — same as OFAT notebook
ARCH_OPTIONS = {
    '(16,)'        : (16,),
    '(32,)'        : (32,),
    '(64,)'        : (64,),
    '(128,)'       : (128,),
    '(32, 16)'     : (32, 16),
    '(64, 32)'     : (64, 32),
    '(128, 64)'    : (128, 64),
    '(64, 32, 16)' : (64, 32, 16),
    '(128, 64, 32)': (128, 64, 32),
}


def objective(trial):
    # --- Architecture ---
    arch_key = trial.suggest_categorical('arch', list(ARCH_OPTIONS.keys()))
    arch = ARCH_OPTIONS[arch_key]

    # --- Activation ---
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'logistic'])

    # --- Solver ---
    solver = trial.suggest_categorical('solver', ['lbfgs', 'adam'])

    # --- L2 regularisation (log-uniform over 6 decades) ---
    alpha = trial.suggest_float('alpha', 1e-5, 10.0, log=True)

    # --- Solver-specific params ---
    if solver == 'adam':
        lr = trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True)
        solver_kwargs = dict(solver='adam', learning_rate_init=lr,
                             max_iter=2000, early_stopping=False)
    else:  # lbfgs
        solver_kwargs = dict(solver='lbfgs', max_iter=2000)

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_layer_sizes=arch,
            activation=activation,
            alpha=alpha,
            random_state=SEED,
            **solver_kwargs,
        )),
    ])

    yp = cross_val_predict(pipe, Xtr, ytr, cv=5)
    return nmad(ytr, yp)   # Optuna minimises this


print('Objective function defined.')

---
## Run the Optuna Study

We run **100 trials**. Optuna automatically:
- Uses TPE (Tree-structured Parzen Estimator) as the sampler
- Explores broadly for the first ~25 trials, then focuses on promising regions
- Handles the conditional `learning_rate_init` parameter correctly
- Tracks every trial for post-hoc analysis

Runtime on CPU: roughly 2–5 minutes depending on hardware.

In [ ]:
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction='minimize', sampler=sampler,
                             study_name='MLP_ZTF_SNIa')

study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nStudy complete — {len(study.trials)} trials evaluated.')
print(f'Best CV NMAD : {study.best_value:.4f}')
print('Best params  :')
for k, v in study.best_params.items():
    print(f'  {k:<30s} = {v}')

---
## Visualising the Optimisation

These plots are one of the biggest advantages of Optuna over a plain grid/random search:
they let you understand *why* certain parameter regions are good, not just *that* they are.

- **Optimisation history**: shows how the best NMAD improved over trials
- **Parameter importances**: which hyperparameters matter most (using fANOVA)
- **Parallel coordinates**: shows the joint parameter combinations for top trials

In [ ]:
# --- Optimisation history ---
ax = plot_optimization_history(study)
ax.set_title('Optuna Optimisation History — CV NMAD per Trial')
ax.set_ylabel('CV NMAD')
plt.tight_layout()
plt.show()

In [ ]:
# --- Parameter importances (fANOVA) ---
# fANOVA decomposes variance in NMAD across trials to attribute it to each parameter.
# High importance = that parameter causes large swings in performance.
ax = plot_param_importances(study)
ax.set_title('Hyperparameter Importances (fANOVA) — Which Knobs Matter Most?')
plt.tight_layout()
plt.show()

In [ ]:
# --- Parallel coordinate plot for top-20 trials ---
# Each line is one trial. Lines clustered together = a good region of parameter space.
top_trials = sorted(study.trials, key=lambda t: t.value)[:20]
top_study = optuna.create_study(direction='minimize')
for t in top_trials:
    top_study.add_trial(t)

ax = plot_parallel_coordinate(top_study)
ax.set_title('Parallel Coordinates — Top-20 Trials')
plt.tight_layout()
plt.show()

In [ ]:
# --- Contour: alpha vs arch (two most physically interpretable params) ---
# Shows how NMAD varies jointly across these two parameters.
try:
    ax = plot_contour(study, params=['alpha', 'arch'])
    ax.set_title('Contour Plot — alpha vs Architecture')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Contour plot skipped: {e}')

In [ ]:
# --- Trial dataframe: top 15 ---
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values('value').head(15)
display_cols = ['number', 'value'] + [c for c in trials_df.columns if c.startswith('params_')]
print('Top-15 trials by CV NMAD:')
print(trials_df[display_cols].to_string(index=False))

---
## Final Model: Test-Set Evaluation

We now build the best-found configuration, refit on the full training set, and
evaluate **once** on the held-out test set. This is the number to report.

> The test set was never touched during optimisation — Optuna only saw CV scores
> on the training fold.

In [ ]:
bp = study.best_params
best_arch = ARCH_OPTIONS[bp['arch']]

if bp['solver'] == 'adam':
    final_solver_kwargs = dict(
        solver='adam',
        learning_rate_init=bp['learning_rate_init'],
        max_iter=2000,
        early_stopping=False,
    )
else:
    final_solver_kwargs = dict(solver='lbfgs', max_iter=2000)

final_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        hidden_layer_sizes=best_arch,
        activation=bp['activation'],
        alpha=bp['alpha'],
        random_state=SEED,
        **final_solver_kwargs,
    )),
])

final_pipe.fit(Xtr, ytr)
yte_pred = final_pipe.predict(Xte)
ytr_pred = final_pipe.predict(Xtr)

test_nmad    = nmad(yte, yte_pred)
test_outlier = outlier_rate(yte, yte_pred)
test_mae     = mean_absolute_error(yte, yte_pred)
test_rmse    = float(np.sqrt(mean_squared_error(yte, yte_pred)))
test_r2      = r2_score(yte, yte_pred)
train_r2     = r2_score(ytr, ytr_pred)

print('=' * 55)
print('  OPTUNA BEST MODEL — TEST-SET RESULTS')
print('=' * 55)
print(f'  Architecture : {best_arch}')
print(f'  Activation   : {bp["activation"]}')
print(f'  Solver       : {bp["solver"]}')
print(f'  alpha        : {bp["alpha"]:.4g}')
if bp['solver'] == 'adam':
    print(f'  LR init      : {bp["learning_rate_init"]:.4g}')
print('-' * 55)
print(f'  CV NMAD      : {study.best_value:.4f} mag')
print(f'  Test NMAD    : {test_nmad:.4f} mag')
print(f'  Outlier Rate : {test_outlier:.1f} %')
print(f'  Test MAE     : {test_mae:.4f} mag')
print(f'  Test RMSE    : {test_rmse:.4f} mag')
print(f'  R² (test)    : {test_r2:.4f}')
print(f'  R² (train)   : {train_r2:.4f}')
print(f'  ΔR² overfit  : {train_r2 - test_r2:.4f}')
print('=' * 55)

In [ ]:
# --- Predicted vs Actual ---
yt = np.array(yte)
yp = np.array(yte_pred)
lo = min(yt.min(), yp.min()) - 0.05
hi = max(yt.max(), yp.max()) + 0.05

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(yt, yp, c=np.array(yerr_te), cmap='plasma', alpha=0.7, s=25)
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect prediction')
ax.set_xlabel(r'Actual $\Delta\mu$')
ax.set_ylabel(r'Predicted $\Delta\mu$')
ax.set_title(f'Optuna MLP — Predicted vs Actual\nNMAD={test_nmad:.4f}  R²={test_r2:.4f}')
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label=r'$\sigma_{\mu,\mathrm{meas}}$')
plt.tight_layout()
plt.show()

In [ ]:
# --- Permutation Feature Importance ---
perm = permutation_importance(
    final_pipe, Xte, yte,
    n_repeats=30, random_state=SEED,
    scoring='neg_mean_absolute_error',
)
fi = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=True)
fi_std = pd.Series(perm.importances_std, index=feature_cols).loc[fi.index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi.index, fi.values, xerr=fi_std.values, color='steelblue',
        alpha=0.8, edgecolor='white', capsize=4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean decrease in MAE when feature shuffled')
ax.set_title('Permutation Feature Importance — Optuna-tuned MLP')
plt.tight_layout()
plt.show()

print('Feature importance (descending):')
for feat, imp in fi.sort_values(ascending=False).items():
    print(f'  {feat:20s}  {imp:.4f}')

---
## Summary

| Aspect | Optuna (this notebook) | OFAT + RandomizedSearchCV |
|---|---|---|
| Search strategy | TPE Bayesian optimisation | Sequential greedy + random |
| Interaction effects | Captured jointly | Partially (Phase 6 only) |
| Sample efficiency | High — focuses on good regions | Moderate — random draws |
| Interpretability | fANOVA importances + contour plots | Phase-sweep plots |
| Trials needed | ~60–100 | ~60 random + 5 sequential phases |

The `plot_param_importances` result is directly comparable to the OFAT notebook's
manual phase-by-phase analysis — if they agree on which parameters matter, that's
a reassuring consistency check. If they disagree, the Optuna fANOVA result
(which sees the joint space) is more trustworthy.